# Montreal Crime After COVID: Trends, Recovery and Hotspots (2015 - 2025)
*Nada Afiri*

Analysis of the crime reported to the Montreal Police Department (SPVM): what happened after the COVID-19 shock, and has it returned to its 2019 level?

## Setup
The first cell imports the helpers from `crime_utils.py`, the second one loads and cleans the data. If `data/actes-criminels.csv` is missing, run `python download_data.py` first.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

from crime_utils import (ALL_CATEGORIES, MAIN_CATEGORIES, WEEKDAYS, TIMES_OF_DAY,
                         annual_variability, cagr, cell_changes, change_map, compare_years,
                         concentration_by_year, data_quality, getis_ord_gi_star, grid_sensitivity,
                         interrupted_time_series, index_to_baseline, last_complete_month,
                         last_covered_month, load_boundary, load_crimes, load_population,
                         missing_location_share, monthly_counts, quality_report, rate_with_ci,
                         same_month_change, weekday_shares, year_month_matrix, year_over_year,
                         year_to_date, yearly_table)

pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype+notebook_connected"

BASE_YEAR = 2019             # last "normal" year before COVID-19
LAST_FULL_YEAR = 2025        # 2026 is only used in the year-to-date comparison
COVID_BREAK = "2020-03-01"   # first full month of the first lockdown (section 6)
CELL_M = 1000                # side of the grid cells, in metres (section 8)
DATA_PATH = "data/actes-criminels.csv"
BOUNDARY_PATH = "data/montreal_boundary.geojson"
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

In [2]:
boundary = load_boundary(BOUNDARY_PATH)
if boundary is None:
    print("No territory polygon found (" + BOUNDARY_PATH + "): coordinates are checked against a "
          "bounding box only.\nRun `python download_data.py` to add the polygon check.")

df = load_crimes(DATA_PATH, boundary=boundary)
full = df[df["YEAR"] <= LAST_FULL_YEAR]          # complete years only
yearly = yearly_table(full)
population = load_population()                   # years without a population are left out
peak_year = int(yearly["Vehicle theft"].idxmax())

## 1. Executive summary
The figures below are computed from the data each time the notebook runs. Details, tests and caveats are in the sections that follow.

In [3]:
total = yearly.sum(axis=1)
vt = yearly["Vehicle theft"]
change = ((yearly.loc[LAST_FULL_YEAR] / yearly.loc[BASE_YEAR] - 1) * 100)[MAIN_CATEGORIES]
conc = concentration_by_year(full)
cut_year, cut_month = last_complete_month(df)
ytd = year_to_date(df, cut_month)
latest = int(df["YEAR"].max())
ytd_total = ytd.sum(axis=1)
its = interrupted_time_series(monthly_counts(full), COVID_BREAK)["table"].loc["Level change at the break"]

lines = []
lines.append("**Volume.** {:,} incidents were reported in {} against {:,} in {} ({:+.1%}). "
             "The lowest year is {} ({:,}).".format(
                 total[LAST_FULL_YEAR], LAST_FULL_YEAR, total[BASE_YEAR], BASE_YEAR,
                 total[LAST_FULL_YEAR] / total[BASE_YEAR] - 1, total.idxmin(), total.min()))
if BASE_YEAR in population.index and LAST_FULL_YEAR in population.index:
    rates = total / population.reindex(total.index) * 100_000
    lines.append("**Per 100,000 residents** the rate went from {:.0f} to {:.0f} ({:+.1%}).".format(
        rates[BASE_YEAR], rates[LAST_FULL_YEAR], rates[LAST_FULL_YEAR] / rates[BASE_YEAR] - 1))
else:
    lines.append("**Per 100,000 residents:** not computed, the population of {} or {} is missing "
                 "in `reference/population_montreal.csv`.".format(BASE_YEAR, LAST_FULL_YEAR))
lines.append("**Vehicle theft** is the category that moves differently: {:,} in {}, a peak of {:,} in {} "
             "({:+.0%} vs {}), then {:,} in {} ({:+.0%} vs {}).".format(
                 vt[BASE_YEAR], BASE_YEAR, vt[peak_year], peak_year, vt[peak_year] / vt[BASE_YEAR] - 1,
                 BASE_YEAR, vt[LAST_FULL_YEAR], LAST_FULL_YEAR, vt[LAST_FULL_YEAR] / vt[BASE_YEAR] - 1, BASE_YEAR))
lines.append("**Other main categories in {} vs {}:** {}.".format(
    LAST_FULL_YEAR, BASE_YEAR,
    ", ".join("{} {:+.1f}%".format(c.lower(), v) for c, v in change.drop("Vehicle theft").items())))
if its["LOW (%)"] <= 0 <= its["HIGH (%)"]:
    covid_text = ("**COVID period.** Total monthly incidents show no clear level change at {} "
                  "({:+.1f}%, 95% CI {:+.1f}% to {:+.1f}%) once trend and seasonality are accounted for; "
                  "categories differ (section 6).")
else:
    covid_text = ("**COVID period.** Total monthly incidents show a level change of {1:+.1f}% "
                  "(95% CI {2:+.1f}% to {3:+.1f}%) from {0}, after accounting for trend and seasonality.")
lines.append(covid_text.format(COVID_BREAK[:7], its["EFFECT (%)"], its["LOW (%)"], its["HIGH (%)"])
             + " This is a temporal association, not proof that the pandemic caused it.")
lines.append("**Concentration.** The busiest 1% of intersections hold {:.1f}% of located incidents in {} "
             "({:.1f}% in {}, {:.1f}% in {}).".format(
                 conc.loc[LAST_FULL_YEAR, "TOP 1% OF LOCATIONS"], LAST_FULL_YEAR,
                 conc.loc[BASE_YEAR, "TOP 1% OF LOCATIONS"], BASE_YEAR,
                 conc.loc[peak_year, "TOP 1% OF LOCATIONS"], peak_year))
lines.append("**{} so far** (January to {}): {:+.1f}% vs {}, {:+.1f}% vs {}.".format(
    latest, pd.Timestamp(year=2000, month=cut_month, day=1).strftime("%B"),
    (ytd_total[latest] / ytd_total[latest - 1] - 1) * 100, latest - 1,
    (ytd_total[latest] / ytd_total[BASE_YEAR] - 1) * 100, BASE_YEAR))
miss_last = missing_location_share(full, "YEAR", "CATEGORY").loc[LAST_FULL_YEAR, MAIN_CATEGORIES]
lines.append("**Caveat.** The data counts incidents *reported* to the police; the date and time of day are those of "
             "the report. {:.1f}% of incidents have no usable position, from {:.0f}% ({}) to {:.0f}% ({}) "
             "by category in {}, so maps are less complete for some crime types.".format(
                 (~df["HAS_LOCATION"]).mean() * 100, miss_last.min(), miss_last.idxmin().lower(),
                 miss_last.max(), miss_last.idxmax().lower(), LAST_FULL_YEAR))
display(Markdown("\n".join("* " + line for line in lines)))

* **Volume.** 29,832 incidents were reported in 2025 against 27,155 in 2019 (+9.9%). The lowest year is 2020 (25,460).
* **Per 100,000 residents** the rate went from 1319 to 1373 (+4.1%).
* **Vehicle theft** is the category that moves differently: 4,170 in 2019, a peak of 11,528 in 2023 (+176% vs 2019), then 6,579 in 2025 (+58% vs 2019).
* **Other main categories in 2025 vs 2019:** vehicle contents or parts theft +0.7%, burglary -4.7%, mischief +4.0%, robbery +15.7%.
* **COVID period.** Total monthly incidents show no clear level change at 2020-03 (-1.5%, 95% CI -17.5% to +17.7%) once trend and seasonality are accounted for; categories differ (section 6). This is a temporal association, not proof that the pandemic caused it.
* **Concentration.** The busiest 1% of intersections hold 9.8% of located incidents in 2025 (7.8% in 2019, 11.1% in 2023).
* **2026 so far** (January to October): -2.8% vs 2025, +8.6% vs 2019.
* **Caveat.** The data counts incidents *reported* to the police; the date and time of day are those of the report. 16.7% of incidents have no usable position, from 7% (burglary) to 22% (vehicle contents or parts theft) by category in 2025, so maps are less complete for some crime types.

## 2. Research questions
1. Compared with 2019, which crime types are higher or lower in 2025, in counts and per 100,000 residents, and are the differences larger than random variation?
2. How large is the vehicle theft surge, when did it start, and when are these thefts reported?
3. Which parts of the island account for the largest increases in vehicle theft, and do they persist when the grid changes?
4. Can a change of level or trend be seen around the start of COVID-19, and does it last?
5. Do crimes follow different weekly and night-time reporting patterns, and have they shifted?
6. Are incidents concentrated in a few locations, and is that changing?
7. How does 2026 so far compare with the same months of earlier years?

**Working hypotheses, checked against the results below.** (a) Crime fell in 2020 and came back unevenly rather than uniformly. (b) Vehicle theft behaves differently from the other categories. (c) Increases are concentrated in a few places rather than spread over the island.

## 3. Data sources
[Montreal Open Data Portal, "Actes criminels"](https://donnees.montreal.ca/dataset/actes-criminels), published by the SPVM under the CC-BY 4.0 licence and updated daily. Each row is one incident reported to the police:

| Column | Official description | Used as |
| --- | --- | --- |
| `CATEGORIE` | Nature of the event (6 values, translated in `crime_utils.CATEGORY_MAP`) | category |
| `DATE` | Date the event was **reported** to the SPVM | year, month, weekday |
| `QUART` | Time of day (day, evening, night) of the **report** | time of day |
| `PDQ` | Neighbourhood police station covering the area | PDQ |
| `LATITUDE`, `LONGITUDE` | Position moved to an intersection (WGS 84) | position |

Because `DATE` and `QUART` describe the report and not necessarily the event, weekday and time-of-day results are **reporting patterns**. Offenses causing death include manslaughter and criminal negligence, not only murder.

Population (for rates per 100,000 residents) comes from the Institut de la statistique du Québec, administrative region of Montréal (the island), in `reference/population_montreal.csv`. The territory boundary (to check coordinates) comes from the portal's administrative boundaries of the agglomeration.

## 4. Data quality
Before analysing anything, here is what the cleaning keeps, drops or flags in the raw file. Nothing is removed silently: rows are only dropped when their date or category cannot be read, and exact duplicate rows are counted but kept, because the file has no incident identifier and two different incidents can share a date, a category and an intersection.

In [4]:
quality_report(DATA_PATH, boundary=boundary)

,Rows,% of raw rows
Check,,
Raw rows,366066,100.00
Invalid dates (dropped),0,0.00
Unrecognised categories (dropped),0,0.00
Rows kept after cleaning,366066,100.00
"Exact duplicate rows (kept, counted only)",11314,3.09
"Missing coordinates (kept, no map position)",61168,16.71
"Coordinates outside the Montreal box (kept, no map position)",0,0.00
"Coordinates outside the territory polygon (kept, no map position)",2,0.00
Missing PDQ (kept),8,0.00


In [5]:
data_quality(df)

,Value
Records,"366,066"
First date,2015-01-01
Last date,2026-11-22
Without usable location,16.7%
Without PDQ,0.00%
Without time of day,0.00%


### Missing locations
About 17% of the incidents have no usable position, so the geographic analysis (section 8) only covers the others. That is safe only if the share of missing positions is similar across years, crime types, months and police stations. If it were not, a change in the number of mapped incidents could come from a change in the missing share, not from a change in crime.

In [6]:
missing_year = missing_location_share(full, "YEAR", "CATEGORY")
display(missing_location_share(full, "YEAR").T)
display(missing_year)

long_missing = (missing_year[MAIN_CATEGORIES].reset_index()
                .melt(id_vars="YEAR", var_name="CATEGORY", value_name="% WITHOUT LOCATION"))
fig = px.line(long_missing, x="YEAR", y="% WITHOUT LOCATION", color="CATEGORY", markers=True,
              title="<b>Incidents without a usable location (%)</b>")
fig.update_xaxes(dtick=1, title=None)
fig.update_yaxes(rangemode="tozero")
fig.show()

YEAR,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
% WITHOUT LOCATION,18.1,17.1,17.1,18.0,17.7,15.9,15.9,15.7,16.1,16.9,16.2


CATEGORY,Burglary,Mischief,Offenses causing death,Robbery,Vehicle contents or parts theft,Vehicle theft
YEAR,,,,,,
2015,3.6,17.8,18.5,25.1,29.2,20.4
2016,3.2,18.2,13.0,24.1,27.2,19.4
2017,4.1,18.1,19.2,22.9,27.6,17.5
2018,5.1,18.9,22.6,19.3,27.4,19.5
2019,3.9,18.2,16.7,21.8,28.3,18.3
2020,5.3,16.3,15.4,19.3,23.2,16.6
2021,5.9,16.9,23.5,19.0,21.6,15.4
2022,7.0,15.8,16.2,15.9,21.7,15.8
2023,7.5,16.3,12.0,17.2,21.7,16.5


In [7]:
by_month = missing_location_share(full, "MONTH")
by_month.index = MONTHS
display(by_month.T)

latest_full = full[full["YEAR"] == LAST_FULL_YEAR]
by_pdq = (missing_location_share(latest_full, "PDQ")
          .join(latest_full.groupby("PDQ").size().rename("INCIDENTS"))
          .sort_values("% WITHOUT LOCATION", ascending=False))
print("Police stations (PDQ) with the highest share of missing positions in {}:".format(LAST_FULL_YEAR))
display(by_pdq.head(8))

miss_cat = missing_year.loc[LAST_FULL_YEAR, MAIN_CATEGORIES]
print("\nIn {} the share of missing positions goes from {:.1f}% ({}) to {:.1f}% ({}); "
      "for vehicle theft it was {:.1f}% in {} and {:.1f}% in {}.".format(
          LAST_FULL_YEAR, miss_cat.min(), miss_cat.idxmin().lower(), miss_cat.max(), miss_cat.idxmax().lower(),
          missing_year.loc[BASE_YEAR, "Vehicle theft"], BASE_YEAR,
          missing_year.loc[peak_year, "Vehicle theft"], peak_year))

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
% WITHOUT LOCATION,15.8,16.3,16.6,16.4,17.2,17.7,17.2,17.1,17.7,17.0,16.6,15.6


Police stations (PDQ) with the highest share of missing positions in 2025:


,% WITHOUT LOCATION,INCIDENTS
PDQ,,
20,30.5,1829
21,29.2,3343
39,20.3,1420
26,18.4,1702
13,16.8,841
30,16.3,765
45,15.7,604
44,15.5,1085



In 2025 the share of missing positions goes from 7.3% (burglary) to 22.1% (vehicle contents or parts theft); for vehicle theft it was 18.3% in 2019 and 16.5% in 2023.


**How to read it.** Positions are *not* missing at random: the share differs a lot between crime types and between police stations, and some lines drift over time. Comparisons between years for one category stay reasonable when its own share is stable (vehicle theft is the category the hotspot analysis rests on, see the sentence above). Comparisons between categories, and maps for stations with many missing positions, must be read with that bias in mind: those areas are *under-counted* on the maps, so the real concentration there is larger than shown.

In [8]:
pdq_per_year = full.groupby("YEAR")["PDQ"].nunique()
print("Number of PDQ: {} in {}, {} in {}".format(
    pdq_per_year.iloc[0], pdq_per_year.index[0], pdq_per_year.iloc[-1], pdq_per_year.index[-1]))
pdq_per_year.to_frame("PDQ count").T

Number of PDQ: 34 in 2015, 29 in 2025


YEAR,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
PDQ count,34,34,34,34,33,32,32,31,31,31,29


## 5. Overall trends
Raw counts hide the story because the categories have very different sizes. I rescale every category so that **2019 = 100**: a value of 130 means 30% more incidents than in 2019. Offenses causing death are left out of this chart (about 30 per year, too few for percentages) and are reported in the tables.

In [9]:
yearly

CATEGORY,Vehicle contents or parts theft,Burglary,Mischief,Vehicle theft,Robbery,Offenses causing death
YEAR,,,,,,
2015,10915,9796,8356,4418,2097,27
2016,10209,9797,7405,4352,1974,23
2017,9556,9197,7302,4732,1851,26
2018,8666,7429,6432,4237,1592,31
2019,8018,6987,6171,4170,1785,24
2020,7403,6057,5869,4706,1399,26
2021,7962,5111,5820,6440,1456,34
2022,9150,6292,6070,9377,1699,37
2023,8198,6541,6682,11528,1992,25


In [10]:
index = index_to_baseline(yearly[MAIN_CATEGORIES], BASE_YEAR)
long_index = index.reset_index().melt(id_vars="YEAR", var_name="CATEGORY", value_name="INDEX")

fig = px.line(long_index, x="YEAR", y="INDEX", color="CATEGORY", markers=True,
              title="<b>Incidents by crime type (2019 = 100)</b>")
fig.add_hline(y=100, line_dash="dot", line_color="grey")
fig.update_xaxes(dtick=1, title=None)
fig.update_yaxes(title="Index")
fig.show()

### 2019 against 2025, with uncertainty
`CHANGE (%)` is the change in the number of incidents; the interval treats the counts as Poisson variables and assumes the same population in both years. When the population of both years is available, the `RATE` columns give the change per resident. `CAGR` is the compound annual growth rate between the two years. An interval that excludes 0 means the difference is larger than random variation alone, which does not prove that the underlying crime changed (reporting habits, policing and other factors also move these counts).

In [11]:
with_total = yearly.assign(**{"All categories": yearly.sum(axis=1)})
comparison = compare_years(with_total, BASE_YEAR, LAST_FULL_YEAR, population)
comparison["CAGR (%)"] = cagr(with_total, BASE_YEAR, LAST_FULL_YEAR).round(1)
comparison.sort_values("CHANGE (%)", ascending=False)

,2019,2025,CHANGE (%),CI LOW (%),CI HIGH (%),RATE CHANGE (%),RATE CI LOW (%),RATE CI HIGH (%),CAGR (%)
Offenses causing death,24,39,62.5,-2.3,170.2,54.0,-7.4,156.1,8.4
Vehicle theft,4170,6579,57.8,51.8,64.0,49.6,43.9,55.5,7.9
Robbery,1785,2066,15.7,8.6,23.3,9.7,3.0,16.9,2.5
All categories,27155,29832,9.9,8.1,11.7,4.1,2.4,5.9,1.6
Mischief,6171,6416,4.0,0.4,7.7,-1.4,-4.8,2.1,0.7
Vehicle contents or parts theft,8018,8076,0.7,-2.3,3.9,-4.5,-7.4,-1.5,0.1
Burglary,6987,6656,-4.7,-7.9,-1.5,-9.7,-12.7,-6.6,-0.8


### Year-over-year change (%)

In [12]:
year_over_year(with_total).round(1)

CATEGORY,Vehicle contents or parts theft,Burglary,Mischief,Vehicle theft,Robbery,Offenses causing death,All categories
YEAR,,,,,,,
2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016,-6.5,0.0,-11.4,-1.5,-5.9,-14.8,-5.2
2017,-6.4,-6.1,-1.4,8.7,-6.2,13.0,-3.2
2018,-9.3,-19.2,-11.9,-10.5,-14.0,19.2,-13.1
2019,-7.5,-5.9,-4.1,-1.6,12.1,-22.6,-4.3
2020,-7.7,-13.3,-4.9,12.9,-21.6,8.3,-6.2
2021,7.6,-15.6,-0.8,36.8,4.1,30.8,5.4
2022,14.9,23.1,4.3,45.6,16.7,8.8,21.6
2023,-10.4,4.0,10.1,22.9,17.2,-32.4,7.2


### Rate per 100,000 residents
The rate is the number of incidents reported divided by the population, times 100,000. It is a frequency of *reported incidents*, not the probability that a resident becomes a victim. The interval is the exact Poisson interval and treats the population as known, so it understates the true uncertainty.

In [13]:
rate_total = rate_with_ci(yearly.sum(axis=1), population).dropna(subset=["POPULATION"])
if rate_total.empty:
    print("No population in reference/population_montreal.csv: rates cannot be computed.")
else:
    display(rate_total.round(1))
    fig = go.Figure(go.Scatter(
        x=rate_total.index, y=rate_total["RATE"], mode="lines+markers", name="Rate",
        error_y=dict(type="data", symmetric=False,
                     array=rate_total["HIGH"] - rate_total["RATE"],
                     arrayminus=rate_total["RATE"] - rate_total["LOW"])))
    fig.update_layout(title="<b>Reported incidents per 100,000 residents</b>", yaxis_title="Rate")
    fig.update_xaxes(dtick=1)
    fig.show()
    missing_years = sorted(set(yearly.index) - set(rate_total.index))
    if missing_years:
        print("Years without a population in reference/population_montreal.csv:", missing_years)

,INCIDENTS,POPULATION,RATE,LOW,HIGH
YEAR,,,,,
2015,35609,1951052.0,1825.1,1806.2,1844.2
2016,33760,1960740.0,1721.8,1703.5,1740.3
2017,32664,1983795.0,1646.5,1628.7,1664.5
2018,28387,2023837.0,1402.6,1386.4,1419.0
2019,27155,2059126.0,1318.8,1303.1,1334.5
2020,25460,2061967.0,1234.7,1219.6,1250.0
2021,26823,2015879.0,1330.6,1314.7,1346.6
2022,32625,2034638.0,1603.5,1586.1,1621.0
2023,34966,2094820.0,1669.2,1651.7,1686.8


### Year-to-year variability
How much each category moves from one year to the next, before the pandemic and recently. A large coefficient of variation (CV) means that a single year's change is less informative.

In [14]:
pd.concat({
    "2015-2019": annual_variability(yearly[MAIN_CATEGORIES], range(2015, 2020)),
    "2022-2025": annual_variability(yearly[MAIN_CATEGORIES], range(2022, 2026)),
}, axis=1)

2015-2019                                  \
                                     MEAN     STD CV (%) LARGEST RISE (%)   
CATEGORY                                                                    
Vehicle contents or parts theft    9472.8  1161.5   12.3             -6.4   
Burglary                           8641.2  1340.2   15.5              0.0   
Mischief                           7133.2   868.2   12.2             -1.4   
Vehicle theft                      4381.8   218.3    5.0              8.7   
Robbery                            1859.8   191.5   10.3             12.1   

                                                 2022-2025                 \
                                LARGEST FALL (%)      MEAN     STD CV (%)   
CATEGORY                                                                    
Vehicle contents or parts theft             -9.3    8533.2   494.2    5.8   
Burglary                                   -19.2    6455.2   172.7    2.7   
Mischief                                   -11.9    6600.0   490.2    7.4   
Vehicle theft                              -10.5    9036.2  2041.6   22.6   
Robbery                                    -14.0    1912.8   159.0    8.3   

                                                                   
                                LARGEST RISE (%) LARGEST FALL (%)  
CATEGORY                                                           
Vehicle contents or parts theft              6.2            -10.4  
Burglary                                     5.1             -3.2  
Mischief                                    10.1            -11.3  
Vehicle theft                               22.9            -24.9  
Robbery                                     17.2             -4.9

## 6. COVID analysis
First a simple comparison of average yearly counts across three periods, indexed on 2015-2019 (= 100).

In [15]:
periods = {"2015-2019": range(2015, 2020), "2020-2021": range(2020, 2022), "2022-2025": range(2022, 2026)}
averages = pd.DataFrame({name: yearly.loc[list(years)].mean() for name, years in periods.items()}).loc[MAIN_CATEGORIES]
(averages.div(averages["2015-2019"], axis=0) * 100).round(1)

,2015-2019,2020-2021,2022-2025
CATEGORY,,,
Vehicle contents or parts theft,100.0,81.1,90.1
Burglary,100.0,64.6,74.7
Mischief,100.0,81.9,92.5
Vehicle theft,100.0,127.2,206.2
Robbery,100.0,76.8,102.8


### Interrupted time series
A segmented regression on the monthly counts estimates the trend before March 2020, the **level change** at the break and the **change of slope** afterwards, with a monthly seasonal pattern and standard errors that are robust to autocorrelation. It works on the logarithm of the counts, so the effects are read as percentages.

A single break cannot separate the lockdowns from the recovery that followed, and the results describe a temporal association with the break date, not the effect of the pandemic itself.

In [16]:
monthly_total = monthly_counts(full)
its_total = interrupted_time_series(monthly_total, COVID_BREAK)
display(its_total["table"].round(3))
print("Durbin-Watson of the residuals: {:.2f} (about 2 means little leftover autocorrelation)".format(
    its_total["durbin_watson"]))
if its_total["durbin_watson"] < 1.5:
    print("Strong leftover autocorrelation: the standard errors account for it, but a straight-line trend "
          "is a rough description of this series, so read the intervals as indicative.")

fig = go.Figure()
fig.add_scatter(x=monthly_total.index, y=monthly_total, name="Observed", mode="lines", line=dict(width=1))
fig.add_scatter(x=monthly_total.index, y=its_total["fitted"], name="Fitted", mode="lines")
fig.add_scatter(x=monthly_total.index, y=its_total["counterfactual"], name="Pre-2020 trend continued",
                mode="lines", line=dict(dash="dash"))
fig.add_shape(type="line", x0=COVID_BREAK, x1=COVID_BREAK, y0=0, y1=1, yref="paper",
              line=dict(dash="dot", color="grey"))
fig.update_layout(title="<b>Monthly incidents, all categories</b>", yaxis_title="Incidents per month",
                  legend_title_text="")
fig.show()

,COEF,EFFECT (%),LOW (%),HIGH (%),P-VALUE
Monthly trend before the break,-0.005,-0.533,-0.658,-0.409,0.000
Level change at the break,-0.015,-1.467,-17.516,17.704,0.871
Change of monthly trend after the break,0.009,0.947,0.555,1.340,0.000


Durbin-Watson of the residuals: 0.47 (about 2 means little leftover autocorrelation)
Strong leftover autocorrelation: the standard errors account for it, but a straight-line trend is a rough description of this series, so read the intervals as indicative.


In [17]:
rows = {}
for category in MAIN_CATEGORIES:
    result = interrupted_time_series(monthly_counts(full, category), COVID_BREAK)
    level = result["table"].loc["Level change at the break"]
    slope = result["table"].loc["Change of monthly trend after the break"]
    rows[category] = {
        "LEVEL CHANGE (%)": level["EFFECT (%)"], "LEVEL 95% LOW": level["LOW (%)"],
        "LEVEL 95% HIGH": level["HIGH (%)"], "LEVEL P-VALUE": level["P-VALUE"],
        "SLOPE CHANGE (% / month)": slope["EFFECT (%)"], "SLOPE P-VALUE": slope["P-VALUE"],
        "DURBIN-WATSON": result["durbin_watson"],
    }
pd.DataFrame(rows).T.round(3)

,LEVEL CHANGE (%),LEVEL 95% LOW,LEVEL 95% HIGH,LEVEL P-VALUE,SLOPE CHANGE (% / month),SLOPE P-VALUE,DURBIN-WATSON
Vehicle contents or parts theft,-4.574,-19.684,13.378,0.594,0.792,0.000,1.060
Burglary,-22.151,-29.893,-13.554,0.000,1.096,0.000,0.846
Mischief,-4.507,-13.026,4.846,0.333,0.925,0.000,0.959
Vehicle theft,41.234,-7.512,115.672,0.110,0.776,0.125,0.260
Robbery,-19.589,-32.048,-4.845,0.011,1.177,0.000,1.214


**Reading the table.** A negative level change means monthly counts were lower than the pre-2020 trend would predict at the break; a positive slope change means the trend after the break rises faster than before. Categories whose level and slope changes have opposite signs (a drop followed by a rise) are the ones where a single break is a crude description. Vehicle theft is a poor fit for a March 2020 break: its surge comes later (see section 7), which is why its level change is imprecise.

## 7. Vehicle theft
Vehicle theft is the category that moves differently from the others. Here is its monthly evolution, smoothed with a 12-month rolling average to show the trend beyond seasonal noise.

In [18]:
vehicle = full[full["CATEGORY"] == "Vehicle theft"]
monthly_vt = monthly_counts(full, "Vehicle theft").to_frame("Monthly incidents")
monthly_vt["12-month average"] = monthly_vt["Monthly incidents"].rolling(12).mean()

fig = px.line(monthly_vt, y=["Monthly incidents", "12-month average"],
              title="<b>Vehicle theft per month</b>")
fig.update_layout(legend_title_text="", xaxis_title=None, yaxis_title="Incidents")
fig.show()

peak_month = monthly_vt["Monthly incidents"].idxmax()
print("Peak year: {} ({:,} incidents, {:+.0%} vs {})".format(
    peak_year, yearly.loc[peak_year, "Vehicle theft"],
    yearly.loc[peak_year, "Vehicle theft"] / yearly.loc[BASE_YEAR, "Vehicle theft"] - 1, BASE_YEAR))
print("Busiest month:", peak_month.strftime("%B %Y"))

Peak year: 2023 (11,528 incidents, +176% vs 2019)
Busiest month: August 2023


Are these thefts reported at a different time of day than before? `QUART` is the part of the day of the **report**, so for a car stolen overnight and noticed in the morning the report falls in the day.

In [19]:
tod = (pd.crosstab(vehicle["YEAR"], vehicle["TIME_OF_DAY"])[TIMES_OF_DAY]
         .reset_index().melt(id_vars="YEAR", var_name="TIME OF DAY", value_name="INCIDENTS"))

fig = px.bar(tod, x="YEAR", y="INCIDENTS", color="TIME OF DAY",
             category_orders={"TIME OF DAY": TIMES_OF_DAY},
             title="<b>Vehicle theft by time of day of the report</b>")
fig.update_xaxes(dtick=1, title=None)
fig.show()

## 8. Geographic analysis
Police station (PDQ) boundaries were reorganized during the period (the number of stations drops from 34 to 29, see section 4), so comparing PDQs year to year would be misleading. Instead the island is divided into a regular **square grid in metres** (1 km by 1 km, projection MTM zone 8, the one the SPVM uses for its own X and Y columns). Each cell is labelled with the coordinates of its centre and the PDQ that most often covers it. Only incidents with a usable position are counted, so the real increases are larger than what the maps show.

### 8.1 Largest absolute increases
The comparison is between 2019 and the peak year of vehicle theft.

In [20]:
changes = cell_changes(full, "Vehicle theft", BASE_YEAR, peak_year, top=15, cell_m=CELL_M)
display(changes[["LABEL", BASE_YEAR, peak_year, "CHANGE", "PCT_CHANGE"]].round(0))

fig = px.bar(changes.iloc[::-1], x="CHANGE", y="LABEL", orientation="h",
             title="<b>Largest increases in vehicle theft per grid cell, {} to {}</b>".format(BASE_YEAR, peak_year))
fig.update_yaxes(title=None, type="category")
fig.update_xaxes(title="Additional incidents")
fig.show()

,LABEL,2019,2023,CHANGE,PCT_CHANGE
0,"45.504, -73.555 (PDQ 21)",23,267,244,1061.0
1,"45.504, -73.568 (PDQ 20)",27,220,193,715.0
2,"45.495, -73.581 (PDQ 12)",13,188,175,1346.0
3,"45.513, -73.568 (PDQ 38)",28,195,167,596.0
4,"45.522, -73.581 (PDQ 38)",14,166,152,1086.0
5,"45.459, -73.824 (PDQ 5)",37,189,152,411.0
6,"45.495, -73.568 (PDQ 20)",7,130,123,1757.0
7,"45.459, -73.747 (PDQ 5)",17,118,101,594.0
8,"45.522, -73.568 (PDQ 38)",17,108,91,535.0
9,"45.513, -73.555 (PDQ 21)",25,114,89,356.0


In [21]:
change_map(changes)   # updates docs/map_hotspots.html, a standalone copy of the map

fig = px.scatter_map(
    changes, lat="CENTRE_LAT", lon="CENTRE_LON", size="CHANGE", hover_name="LABEL",
    size_max=30, zoom=10, height=520, color_discrete_sequence=["#b2182b"],
)
fig.update_layout(
    map_style="white-bg",
    map_layers=[{
        "below": "traces", "sourcetype": "raster",
        "source": ["https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"],
        "sourceattribution": "Tiles © Esri",
    }],
    map_center=dict(lat=45.55, lon=-73.65),   # island of Montreal (also the "reset view" target)
    map_zoom=10,
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.show()

*Circles are centered on each cell and sized by the increase.*

**How complete is the map where the increases are?** Share of vehicle thefts without a position, in the police stations (PDQ) that most often cover the top cells. PDQ boundaries changed, so a station may not exist in both years.

In [22]:
vt_located = full[(full["CATEGORY"] == "Vehicle theft") & full["YEAR"].isin([BASE_YEAR, peak_year])]
top_pdqs = pd.Series(changes["MAIN_PDQ"].str.replace("PDQ ", "", regex=False).astype(int).unique())
pdq_missing = missing_location_share(vt_located, "PDQ", "YEAR").reindex(top_pdqs.head(6))
print("Vehicle theft without a position (%), by PDQ of the top cells:")
display(pdq_missing)

Vehicle theft without a position (%), by PDQ of the top cells:


YEAR,2019,2023
21,37.1,22.6
20,48.4,25.2
12,23.1,14.4
38,27.2,19.2
5,7.1,11.9
23,26.1,14.7


Where this share is high, the cell counts understate the real number of thefts, so the increases in those areas are underestimated rather than overstated.

### 8.2 Relative increases
Cells that started from very few incidents can show huge percentages (from 1 to 5 incidents is +400%) for a small volume. The ranking below keeps only cells with at least 20 incidents in 2019.

In [23]:
relative = cell_changes(full, "Vehicle theft", BASE_YEAR, peak_year, top=10, cell_m=CELL_M,
                        sort_by="PCT_CHANGE", min_base=20)
relative[["LABEL", BASE_YEAR, peak_year, "CHANGE", "PCT_CHANGE"]].round(0)

,LABEL,2019,2023,CHANGE,PCT_CHANGE
0,"45.504, -73.555 (PDQ 21)",23,267,244,1061.0
1,"45.504, -73.568 (PDQ 20)",27,220,193,715.0
2,"45.513, -73.568 (PDQ 38)",28,195,167,596.0
3,"45.459, -73.824 (PDQ 5)",37,189,152,411.0
4,"45.513, -73.555 (PDQ 21)",25,114,89,356.0
5,"45.522, -73.606 (PDQ 38)",21,59,38,181.0
6,"45.567, -73.594 (PDQ 30)",20,50,30,150.0
7,"45.504, -73.696 (PDQ 7)",20,49,29,145.0
8,"45.594, -73.568 (PDQ 46)",31,67,36,116.0
9,"45.549, -73.593 (PDQ 35)",21,44,23,110.0


### 8.3 Do the hotspots depend on the grid?
Conclusions about places can depend on how the territory is cut. The table repeats the analysis with cells of 500 m, 1 km and 2 km: how many cells have incidents, how much of the total increase the top 15 cells hold, and how many of those top cells overlap one of the top 15 cells of the 1 km reference grid.

In [24]:
sensitivity = grid_sensitivity(full, "Vehicle theft", BASE_YEAR, peak_year, sizes=(500, 1000, 2000),
                               top=15, reference=CELL_M)
sensitivity

,CELLS WITH INCIDENTS,TOP CELLS' SHARE OF THE INCREASE (%),TOP CELLS OVERLAPPING THE REFERENCE TOP CELLS (%)
CELL SIZE (m),,,
500,1478,16.3,80.0
1000,480,29.1,100.0
2000,149,46.8,53.3


### 8.4 Statistically significant hotspots (Getis-Ord Gi*)
The maps above show where counts *increased*. Gi* asks a different question: where is the incident count, in a cell and its 8 neighbours, significantly higher than the island average? A cell is a **hotspot** when its z-score is positive and its p-value, adjusted for the large number of cells tested (Benjamini-Hochberg false discovery rate), is below 0.05. The study area is every cell with at least one located incident that year.

In [25]:
gi = getis_ord_gi_star(full, "Vehicle theft", peak_year, cell_m=CELL_M)
gi_base = getis_ord_gi_star(full, "Vehicle theft", BASE_YEAR, cell_m=CELL_M)
print("{}: {} hotspot cells out of {}.   {}: {} hotspot cells out of {}.".format(
    peak_year, int(gi["HOTSPOT"].sum()), len(gi), BASE_YEAR, int(gi_base["HOTSPOT"].sum()), len(gi_base)))
display(gi.head(10)[["CENTRE_LAT", "CENTRE_LON", "COUNT", "Z", "P_FDR"]].round(4))

gi["STATUS"] = np.select([gi["HOTSPOT"], gi["COLDSPOT"]], ["Hotspot", "Coldspot"], "Not significant")
gi["SIZE"] = np.sqrt(gi["COUNT"]) + 1
fig = px.scatter_map(
    gi, lat="CENTRE_LAT", lon="CENTRE_LON", color="STATUS", size="SIZE", size_max=14, zoom=10, height=520,
    hover_data={"COUNT": True, "Z": ":.2f", "SIZE": False},
    color_discrete_map={"Hotspot": "#b2182b", "Coldspot": "#2166ac", "Not significant": "#bbbbbb"},
    category_orders={"STATUS": ["Hotspot", "Coldspot", "Not significant"]},
    title="<b>Gi* hotspots of vehicle theft, {}</b>".format(peak_year),
)
fig.update_layout(
    map_style="white-bg",
    map_layers=[{
        "below": "traces", "sourcetype": "raster",
        "source": ["https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"],
        "sourceattribution": "Tiles © Esri",
    }],
    map_center=dict(lat=45.55, lon=-73.65), map_zoom=10,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

2023: 23 hotspot cells out of 503.   2019: 61 hotspot cells out of 501.


,CENTRE_LAT,CENTRE_LON,COUNT,Z,P_FDR
0,45.5044,-73.5678,220,14.1322,0.0
1,45.5134,-73.5678,195,13.5528,0.0
2,45.4954,-73.5678,130,10.7887,0.0
3,45.5044,-73.5550,267,9.7265,0.0
4,45.5134,-73.5806,83,9.4851,0.0
5,45.5134,-73.5550,114,9.4610,0.0
6,45.5044,-73.5806,95,9.1954,0.0
7,45.5224,-73.5678,108,8.1212,0.0
8,45.5224,-73.5806,166,7.3728,0.0
9,45.4954,-73.5806,188,7.3004,0.0


Gi* compares each neighbourhood with the island average **of the same year**. The number of hotspot cells is therefore not a measure of how much vehicle theft grew: it depends on how unevenly the incidents are spread that year. Read the maps for *where* the clusters are, and the counts of sections 5 and 8.1 for *how much* it grew.

### 8.5 Concentration of incidents
If a small share of intersections concentrates a large share of incidents, targeted prevention is more effective. For each year: the share of all located incidents at the busiest 1% and 5% of intersections, the **Gini coefficient** (0 = incidents spread evenly over intersections, 1 = all at one place) and the **Herfindahl-Hirschman index** (HHI, sum of squared shares, 0 to 10,000; higher = more concentrated; the lowest possible value is 10,000 divided by the number of intersections, about 1 here).

A place with many incidents is not automatically a place with an unusual concentration: concentration is about the *distribution* of incidents, volume is about the *total*.

In [26]:
conc = concentration_by_year(full)
display(conc.round(2))

long_conc = (conc[["TOP 1% OF LOCATIONS", "TOP 5% OF LOCATIONS"]].reset_index()
                 .melt(id_vars="YEAR", var_name="GROUP", value_name="SHARE OF INCIDENTS (%)"))
fig = px.line(long_conc, x="YEAR", y="SHARE OF INCIDENTS (%)", color="GROUP", markers=True,
              title="<b>Share of incidents at the busiest intersections</b>")
fig.update_xaxes(dtick=1, title=None)
fig.show()

fig = px.line(conc.reset_index(), x="YEAR", y="GINI", markers=True,
              title="<b>Gini coefficient of incidents across intersections</b>")
fig.update_xaxes(dtick=1, title=None)
fig.show()

,LOCATIONS,TOP 1% OF LOCATIONS,TOP 5% OF LOCATIONS,GINI,HHI
YEAR,,,,,
2015,10326,8.42,23.23,0.45,2.33
2016,10147,8.20,23.19,0.44,2.35
2017,9999,8.03,22.26,0.43,2.38
2018,9458,8.18,22.73,0.43,2.42
2019,9161,7.77,21.88,0.41,2.36
2020,9047,7.30,20.85,0.40,2.34
2021,9315,8.24,21.88,0.41,2.52
2022,10007,9.58,24.11,0.45,2.92
2023,10077,11.11,26.65,0.47,3.19


## 9. Temporal patterns
**These are reporting patterns.** `DATE` is the day the incident was reported and `QUART` the part of the day of the report, so the charts show when incidents are *reported to the police*, which can differ from when they happened (a stolen car is often reported the next morning, a burglary when the residents come home). A change in these shares can reflect a change in reporting practice, such as online reporting, as much as a change in behaviour.

### 9.1 Weekday
For each crime type, the share of its reports falling on each weekday.

In [27]:
week = full.pivot_table(index="CATEGORY", columns="WEEKDAY", values="DATE",
                        aggfunc="count", observed=False)[WEEKDAYS]
week = week.div(week.sum(axis=1), axis=0).mul(100).loc[MAIN_CATEGORIES]

fig = px.imshow(week, text_auto=".1f", aspect="auto", color_continuous_scale="Blues",
                labels=dict(x="", y="", color="% of reports"),
                title="<b>Share of each crime type by weekday of the report (%)</b>")
fig.show()

Has the weekday pattern of vehicle theft reports changed?

In [28]:
before = weekday_shares(full, "Vehicle theft", range(2015, 2020))
after = weekday_shares(full, "Vehicle theft", range(2022, 2026))
shift = pd.DataFrame({"2015-2019": before, "2022-2025": after}).round(1)
display(shift.T)

long_shift = shift.reset_index().melt(id_vars="WEEKDAY", var_name="PERIOD", value_name="% OF REPORTS")
fig = px.bar(long_shift, x="WEEKDAY", y="% OF REPORTS", color="PERIOD", barmode="group",
             category_orders={"WEEKDAY": WEEKDAYS},
             title="<b>Vehicle theft: weekday of the report</b>")
fig.update_xaxes(title=None)
fig.show()

WEEKDAY,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
2015-2019,15.7,15.6,16.2,15.8,15.0,11.4,10.2
2022-2025,15.6,14.3,14.1,14.0,14.4,13.3,14.3


A difference between the two periods is a change in *when thefts are reported*. It should not be read as proof that thieves changed their habits.

### 9.2 Night
Share of each crime type reported at night, by year.

In [29]:
night = (pd.crosstab([full["YEAR"], full["CATEGORY"]], full["TIME_OF_DAY"], normalize="index")["night"]
           .mul(100).rename("NIGHT SHARE (%)").reset_index())
night = night[night["CATEGORY"].isin(MAIN_CATEGORIES)]

fig = px.line(night, x="YEAR", y="NIGHT SHARE (%)", color="CATEGORY", markers=True,
              title="<b>Share of reports made at night</b>")
fig.update_xaxes(dtick=1, title=None)
fig.show()

### 9.3 Year and month
Seasonality is separated from the yearly trend by comparing the same months across years. The heatmaps give the number of reports per year and month.

In [30]:
for title, category in [("All categories", None), ("Vehicle theft", "Vehicle theft")]:
    matrix = year_month_matrix(full, category)
    fig = px.imshow(matrix.to_numpy(), x=MONTHS, y=matrix.index.astype(str), text_auto=True, aspect="auto",
                    color_continuous_scale="Blues", labels=dict(x="", y="", color="Incidents"),
                    title="<b>Incidents per year and month: {}</b>".format(title))
    fig.show()

Same month, 2019 against 2025: the change of each calendar month, which avoids confusing seasonality with the yearly trend.

In [31]:
same_month = pd.concat({
    "All categories": same_month_change(full, BASE_YEAR, LAST_FULL_YEAR),
    "Vehicle theft": same_month_change(full, BASE_YEAR, LAST_FULL_YEAR, "Vehicle theft"),
}, axis=1)
same_month.index = MONTHS
same_month

All categories                  Vehicle theft                
              2019  2025 CHANGE (%)          2019 2025 CHANGE (%)
Jan           1839  2317       26.0           251  585      133.1
Feb           1854  1934        4.3           271  479       76.8
Mar           2179  2479       13.8           346  549       58.7
Apr           2354  2736       16.2           420  565       34.5
May           2344  2665       13.7           345  560       62.3
Jun           2265  2711       19.7           356  599       68.3
Jul           2313  2555       10.5           325  587       80.6
Aug           2255  2680       18.8           363  688       89.5
Sep           2603  2592       -0.4           443  545       23.0
Oct           2619  2615       -0.2           364  495       36.0
Nov           2402  2419        0.7           353  452       28.0
Dec           2128  2129        0.0           333  475       42.6

## 10. 2026 update
The file stops in mid-November 2026, so a full-year comparison is impossible. I compare the same window (January to the last complete month) for every year.

The file only says which date has data, not which months the source has finished publishing. A month is therefore treated as complete only when the data extends at least 7 days beyond its end, so a file ending on October 31 would still treat October as incomplete.

In [32]:
cov_year, cov_month = last_covered_month(df)
cut_year, cut_month = last_complete_month(df)
month_name = pd.Timestamp(year=cut_year, month=cut_month, day=1).strftime("%B")
print("Last date in the file:", df["DATE"].max().date())
print("Last month with data: {}-{:02d}.  Last month treated as complete: {}-{:02d}.".format(
    cov_year, cov_month, cut_year, cut_month))
print("Window used: January to {} of each year".format(month_name))

ytd = year_to_date(df, cut_month)
ytd_total = ytd.sum(axis=1).rename("INCIDENTS").reset_index()

fig = px.bar(ytd_total, x="YEAR", y="INCIDENTS", text="INCIDENTS",
             title="<b>Incidents from January to {}</b>".format(month_name))
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_xaxes(dtick=1, title=None)
fig.show()

Last date in the file: 2026-11-22
Last month with data: 2026-11.  Last month treated as complete: 2026-10.
Window used: January to October of each year


In [33]:
latest = int(df["YEAR"].max())
table = pd.DataFrame({
    "vs {}".format(latest - 1): (ytd.loc[latest] / ytd.loc[latest - 1] - 1) * 100,
    "vs {}".format(BASE_YEAR): (ytd.loc[latest] / ytd.loc[BASE_YEAR] - 1) * 100,
}).round(1)
table.loc["All categories"] = [
    round((ytd.loc[latest].sum() / ytd.loc[latest - 1].sum() - 1) * 100, 1),
    round((ytd.loc[latest].sum() / ytd.loc[BASE_YEAR].sum() - 1) * 100, 1),
]
print("Change (%) of {} (January to {}) compared with the same months".format(latest, month_name))
display(table)

ytd_all = ytd.assign(**{"All categories": ytd.sum(axis=1)})
print("{} vs {}, with a 95% interval on the change in the number of incidents:".format(latest, latest - 1))
compare_years(ytd_all, latest - 1, latest)

Change (%) of 2026 (January to October) compared with the same months


,vs 2025,vs 2019
CATEGORY,,
Vehicle contents or parts theft,5.3,5.6
Burglary,-5.3,-7.3
Mischief,-6.6,-1.6
Vehicle theft,-6.9,51.0
Robbery,0.1,20.2
Offenses causing death,-51.4,5.9
All categories,-2.8,8.6


2026 vs 2025, with a 95% interval on the change in the number of incidents:


,2025,2026,CHANGE (%),CI LOW (%),CI HIGH (%)
Vehicle contents or parts theft,6657,7013,5.3,1.9,8.9
Burglary,5652,5352,-5.3,-8.8,-1.7
Mischief,5525,5160,-6.6,-10.1,-3.0
Vehicle theft,5652,5261,-6.9,-10.3,-3.4
Robbery,1761,1762,0.1,-6.3,6.9
Offenses causing death,37,18,-51.4,-72.3,-14.6
All categories,25284,24566,-2.8,-4.5,-1.1


## 11. Limitations
* The data counts incidents **reported to the police**, not all crimes. A change in reporting habits (for example online reporting) can look like a change in crime.
* `DATE` and `QUART` are the date and part of the day of the **report**, so weekday and time-of-day results are reporting patterns, not the pattern of the offences.
* Rates per 100,000 residents use the population of the administrative region of Montréal (the island) as an approximation of the SPVM territory, and they are frequencies of reported incidents, not individual risk.
* Poisson intervals assume independent incidents with a constant rate and treat the population as known; real data is more variable, so the intervals are lower bounds on the uncertainty.
* Five categories are compared at once, with no correction for multiple comparisons: an interval bound close to 0 (mischief) should be read as borderline.
* About 17% of rows have no usable position, and not evenly: the share depends on the crime type and the police station (section 4). Location analyses cover the located incidents only, so areas and categories with many missing positions are under-counted on the maps. Locations are moved to the nearest intersection by the SPVM for privacy.
* PDQ boundaries changed during the period, so the geographic analysis uses a regular grid. Results can depend on the grid; section 8.3 tests it.
* The interrupted time series uses a single break date and describes an association with it; it cannot separate lockdown effects from other changes happening at the same time.
* The fitted line of the interrupted time series overshoots the observed counts in 2024-2025, because a single straight post-2020 trend cannot follow a rise and a fall; read its estimates as indicative.
* Exact duplicate rows are kept: the file has no incident identifier, so they cannot be told apart from distinct incidents.
* "Offenses causing death" includes manslaughter and criminal negligence, not only murder, and is too rare for percentage trends.
* The population series drops by 2.2% in 2021, likely a methodological break after the 2021 census. If so, rates before 2021 are slightly underestimated and the per-resident increase since 2019 slightly overstated.

## 12. Conclusions

In [34]:
vt_comparison = compare_years(yearly[["Vehicle theft"]], BASE_YEAR, LAST_FULL_YEAR, population).loc["Vehicle theft"]
above = comparison.loc[MAIN_CATEGORIES]
significant_up = [c for c in MAIN_CATEGORIES if above.loc[c, "CI LOW (%)"] > 0]
significant_down = [c for c in MAIN_CATEGORIES if above.loc[c, "CI HIGH (%)"] < 0]
top_cell = changes.iloc[0]

text = []
text.append("1. **Where crime stands against 2019.** In {}, the categories with a count above {} by more than "
            "random variation are: {}. Below {} by more than random variation: {}.".format(
                LAST_FULL_YEAR, BASE_YEAR, ", ".join(significant_up).lower() or "none",
                BASE_YEAR, ", ".join(significant_down).lower() or "none"))
text.append("2. **Vehicle theft.** The rise starts in 2020-2021, peaks in {} ({:,} reports) and has fallen since, "
            "but {} is still {:+.1f}% above {} (95% interval {:+.1f}% to {:+.1f}%).".format(
                peak_year, yearly.loc[peak_year, "Vehicle theft"], LAST_FULL_YEAR,
                vt_comparison["CHANGE (%)"], BASE_YEAR, vt_comparison["CI LOW (%)"], vt_comparison["CI HIGH (%)"]))
overlap = "TOP CELLS OVERLAPPING THE REFERENCE TOP CELLS (%)"
text.append("3. **Where.** The largest absolute increase between {} and {} is in the cell centred on {:.3f}, {:.3f} "
            "({:+d} reports). With cells of 500 m, {:.0f}% of the top 15 cells overlap the top 15 of the 1 km grid "
            "({:.0f}% with 2 km cells). Gi* flags {} hotspot cells in {} ({} in {}); this counts clusters, "
            "not growth. Vehicle thefts without a position are {:.0f}% in {} and {:.0f}% in {}, so the maps "
            "understate the real counts.".format(
                BASE_YEAR, peak_year, top_cell["CENTRE_LAT"], top_cell["CENTRE_LON"], int(top_cell["CHANGE"]),
                sensitivity.loc[500, overlap], sensitivity.loc[2000, overlap],
                int(gi["HOTSPOT"].sum()), peak_year, int(gi_base["HOTSPOT"].sum()), BASE_YEAR,
                missing_year.loc[BASE_YEAR, "Vehicle theft"], BASE_YEAR,
                missing_year.loc[peak_year, "Vehicle theft"], peak_year))
text.append("4. **Concentration.** The busiest 1% of intersections hold {:.1f}% of located incidents in {} "
            "against {:.1f}% in {} and {:.1f}% in {} (Gini {:.2f}, {:.2f} and {:.2f}).".format(
                conc.loc[LAST_FULL_YEAR, "TOP 1% OF LOCATIONS"], LAST_FULL_YEAR,
                conc.loc[BASE_YEAR, "TOP 1% OF LOCATIONS"], BASE_YEAR,
                conc.loc[peak_year, "TOP 1% OF LOCATIONS"], peak_year,
                conc.loc[LAST_FULL_YEAR, "GINI"], conc.loc[BASE_YEAR, "GINI"], conc.loc[peak_year, "GINI"]))
text.append("5. **Reporting patterns.** Vehicle theft reports on Saturday and Sunday were {:.1f}% and {:.1f}% in "
            "2015-2019 and {:.1f}% and {:.1f}% in 2022-2025. This is a change in when incidents are reported.".format(
                before["Saturday"], before["Sunday"], after["Saturday"], after["Sunday"]))
text.append("6. **{} so far.** January to {}: {:+.1f}% vs {} and {:+.1f}% vs {}.".format(
    latest, month_name, table.loc["All categories", "vs {}".format(latest - 1)], latest - 1,
    table.loc["All categories", "vs {}".format(BASE_YEAR)], BASE_YEAR))
display(Markdown("\n".join(text)))

1. **Where crime stands against 2019.** In 2025, the categories with a count above 2019 by more than random variation are: mischief, vehicle theft, robbery. Below 2019 by more than random variation: burglary.
2. **Vehicle theft.** The rise starts in 2020-2021, peaks in 2023 (11,528 reports) and has fallen since, but 2025 is still +57.8% above 2019 (95% interval +51.8% to +64.0%).
3. **Where.** The largest absolute increase between 2019 and 2023 is in the cell centred on 45.504, -73.555 (+244 reports). With cells of 500 m, 80% of the top 15 cells overlap the top 15 of the 1 km grid (53% with 2 km cells). Gi* flags 23 hotspot cells in 2023 (61 in 2019); this counts clusters, not growth. Vehicle thefts without a position are 18% in 2019 and 16% in 2023, so the maps understate the real counts.
4. **Concentration.** The busiest 1% of intersections hold 9.8% of located incidents in 2025 against 7.8% in 2019 and 11.1% in 2023 (Gini 0.46, 0.41 and 0.47).
5. **Reporting patterns.** Vehicle theft reports on Saturday and Sunday were 11.4% and 10.2% in 2015-2019 and 13.3% and 14.3% in 2022-2025. This is a change in when incidents are reported.
6. **2026 so far.** January to October: -2.8% vs 2025 and +8.6% vs 2019.

**Recommendations that follow from the analysis.** Keep vehicle theft as the priority category to monitor, since it is the only main category far above its pre-pandemic level. Use rates and intervals, not raw percentages, when comparing years. Treat the hotspot maps as a place to look, then confirm with the Gi* result and the grid test before drawing a conclusion about a neighbourhood. Interpret weekday and night results as reporting patterns until the date of the event is available.

### Export figures for the README (optional)
Saves three figures as images in `images/` (needs the `kaleido` package: `pip install kaleido`). They are used by the project README.

In [35]:
from pathlib import Path

Path("images").mkdir(exist_ok=True)
try:
    long_idx = index_to_baseline(yearly[MAIN_CATEGORIES], BASE_YEAR).reset_index().melt(
        id_vars="YEAR", var_name="CATEGORY", value_name="INDEX")
    f1 = px.line(long_idx, x="YEAR", y="INDEX", color="CATEGORY", markers=True,
                 title="<b>Incidents by crime type (2019 = 100)</b>")
    f1.add_hline(y=100, line_dash="dot", line_color="grey")
    f1.update_xaxes(dtick=1, title=None)
    f1.write_image("images/index_2019.png", width=1000, height=520, scale=2)

    f2 = px.line(monthly_vt, y=["Monthly incidents", "12-month average"], title="<b>Vehicle theft per month</b>")
    f2.update_layout(legend_title_text="", xaxis_title=None, yaxis_title="Incidents")
    f2.write_image("images/vehicle_theft_monthly.png", width=1000, height=520, scale=2)

    f3 = px.bar(changes.iloc[::-1], x="CHANGE", y="LABEL", orientation="h",
                title="<b>Largest increases in vehicle theft per grid cell, {} to {}</b>".format(BASE_YEAR, peak_year))
    f3.update_yaxes(title=None, type="category")
    f3.update_xaxes(title="Additional incidents")
    f3.write_image("images/vehicle_theft_cells.png", width=1000, height=620, scale=2)
    print("Saved 3 images in images/")
except Exception as error:   # kaleido missing or no browser available
    print("Images not saved:", error)

Saved 3 images in images/
